# AI Agents and Workflows for Developers
  Author: Svetoslav Yavorov


# Multi-Agent Medical Symptom Assistant (Exam Project)

This notebook demonstrates a stateful multi-agent workflow built with LangChain and LangGraph. The system extracts symptoms from a user request **(act as Patient)** , performs lightweight background research, and produces a short educational recommendation. A Human-in-the-Loop (HITL) interruption is included to simulate expert review before finalizing output. The structure below is organized to support exam revision by separating configuration, agents/tools, graph construction, and evaluation.

## 0) Setting up the project

*   First add an OpenAI API Key in the secrets tab
*   Then add an LangSmith API Key in the secrets tab
*   Press *Run All* or execute code cells one by one

## 1) Environment Setup (Dependencies)

In [1]:
!pip install -q langchain langchain-openai langchain-community langgraph wikipedia

## 2) Imports, Secrets, and Utility Helpers

This section centralizes core imports, secret loading, and small helper utilities used throughout the workflow. It also configures tracing/observability (LangSmith) so agent behavior can be inspected during development and review.

In [ ]:
import json
import operator
import re
import uuid
import os

from IPython.display import HTML, Image, display
from google.colab import userdata

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool, create_retriever_tool
from langchain_openai import ChatOpenAI

from langchain_community.retrievers import WikipediaRetriever

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, Interrupt, interrupt

from pathlib import Path
from pydantic import SecretStr
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "AI Agents and Workflows for Developers - Exam Project"

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


def print_interrupts(interrupts: List[Interrupt]):
    for el in interrupts:
        display(
            HTML(
                f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{el.value}</div>'
            )
        )


def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph(xray=True).draw_mermaid_png())
    display(Image(output_png, format="png"))


def _extract_json(text: str) -> Dict[str, Any]:
    """Best-effort JSON extraction. (handles ```json fences)."""
    if not text:
        return {}

    cleaned_text = text.strip()
    cleaned_text = re.sub(r"^```(?:json)?\s*", "", cleaned_text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r"```$", "", cleaned_text).strip()


    match = re.search(r"\{.*\}", cleaned_text, flags=re.DOTALL)
    if match:
        cleaned_text = match.group(0)

    try:
        parsed = json.loads(cleaned_text)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}

def _cap_lines(text: str, max_lines: int = 10) -> str:
    lines = [ln.rstrip() for ln in (text or "").strip().splitlines() if ln.strip()]
    return "\n".join(lines[:max_lines]).strip()


## 3) Model Configuration (LLM Clients)

In [ ]:
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

extractor_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")
medical_researcher_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")
reporter_model = ChatOpenAI(model="gpt-5-mini", api_key=openai_api_key, reasoning_effort="low")

## 4) State Definition (LangGraph Contract)

This section defines the explicit state object passed between LangGraph nodes.

In [ ]:
class MedicalState(TypedDict, total=False):
    messages: Annotated[List[BaseMessage], add_messages]
    user_request: str
    intent: Literal["health", "non_health"]
    extracted_data: Dict[str, Any]
    patient_summary: str
    severity: Literal["low", "medium", "high", "emergency"]
    researcher_notes: str
    possible_conditions: List[str]
    medical_report: str
    final_recommendation: str
    medical_expert_decision: Literal["approve", "revise"]
    medical_expert_guidance: str
    log: Annotated[List[str], operator.add]

## 5) Tools

This section defines the external tools that agents can call to augment the language model. Tools provide deterministic behavior and controlled access to external information sources (e.g., Wikipedia retrieval). In LangGraph, tools also create explicit decision points that can be routed conditionally based on pending tool calls.

In [ ]:
@tool
def symptom_severity_tool(symptoms: List[str], red_flags: List[str], age: Optional[int] = None) -> str:
    """Estimate a symptom severity level from symptoms and red flags.

    Returns a JSON string: {severity, actions_needed, red_flags_found}.
    """
    symptoms_normalized = [s.strip().lower() for s in symptoms or []]
    red_flags_normalized = [s.strip().lower() for s in red_flags or []]

    emergency_keywords = [
        "chest pain",
        "severe shortness of breath",
        "cannot breathe",
        "fainting",
        "stroke",
        "confusion",
        "seizure",
        "blue lips",
    ]
    high_attention_keywords = [
        "high fever",
        "stiff neck",
        "severe headache",
        "blood in vomit",
        "blood in stool",
        "severe abdominal pain",
    ]

    red_flags_found = any(any(k in rf for k in emergency_keywords + high_attention_keywords) for rf in red_flags_normalized)
    emergency_found = any(any(k in s for k in emergency_keywords) for s in symptoms_normalized) or any(
        any(k in rf for k in emergency_keywords) for rf in red_flags_normalized
    )
    high_found = any(any(k in s for k in high_attention_keywords) for s in symptoms_normalized) or any(
        any(k in rf for k in high_attention_keywords) for rf in red_flags_normalized
    )

    if emergency_found:
        severity = "emergency"
        actions_needed = "Red-flag symptoms suggest possible emergency; urgent evaluation is recommended."
    elif high_found or (
        age is not None and age >= 65
    ):
        severity = "high"
        actions_needed = "Symptoms may be serious or the person is an elder, which can be a problem ,even symtomps are low evaluation. Same-day medical evaluation is recommended."
    elif any(k in " ".join(symptoms_normalized) for k in ["fever", "vomiting", "rash", "dizziness"]):
        severity = "medium"
        actions_needed = "Symptoms may need monitoring and possible medical advice if worsening."
    else:
        severity = "low"
        actions_needed = "No obvious red flags detected; consider basic self-care and monitoring."

    return json.dumps(
        {
            "severity": severity,
            "actions_needed": actions_needed,
            "red_flag_found": red_flags_found,
        },
        indent=2,
    )

wikipedia_retriever = WikipediaRetriever(
    top_k_results=3,
    lang="en",
    doc_content_chars_max=0
)

_base_wikipedia_tool = create_retriever_tool(
    retriever=wikipedia_retriever,
    name="wikipedia_medical_lookup_internal",
    description="""
Search Wikipedia for concise medical background information.

RULES:
- Query MUST be short (1-4 words)
- Use only symptom names or medical terms
- DO NOT use full sentences
- DO NOT include reasoning phrases like:
  "causes", "when standing", "sudden onset"

GOOD:
- "chest pain"
- "orthostatic hypotension"
- "sore throat fever"

BAD:
- "chest pain sudden onset causes heart attack"
- "dizziness when standing up"

Use this tool only for brief educational context.
""",
    document_separator="\n=====\n",
    document_prompt=PromptTemplate(
        input_variables=["title", "summary"],
        template="Article: {title}\nSummary: {summary}"
    )
)

def _normalize_query(query: str) -> str:
    """
    Normalize overly long or noisy medical queries.
    """

    if not query:
        return ""

    query = query.lower().strip()

    banned_words = {
        "causes",
        "cause",
        "sudden",
        "onset",
        "when",
        "while",
        "during",
        "standing",
        "started",
        "because",
        "due",
        "symptoms",
        "possible",
        "why",
    }

    words = [w for w in query.split() if w not in banned_words]

    return " ".join(words[:4])


## Eventhough ,I try to normalize the query, sometimes the results from wikipedia are empty or are not in JSON format.
## Thats why in the test you can see, "Wikipedia summary temporarily unavailable."
## For sure I can switch from using the retriever to sending API request and handle 204 No contents or other statuses with
## re-try mechanism, but the idea in the course is to show how retrievers can be useful, no how good web developers we are.

@tool
def wikipedia_medical_lookup(query: str) -> str:
    """
    Safe Wikipedia medical lookup tool.

    Returns concise educational summaries without breaking the pipeline.
    """

    try:
        safe_query = _normalize_query(query)

        if not safe_query:
            return "No valid Wikipedia query."

        result = _base_wikipedia_tool.invoke({"query": safe_query})

        if not result or not str(result).strip():
            return "No Wikipedia summary found."

        return result

    except Exception:
        return "Wikipedia summary temporarily unavailable."

EXTRACTOR_TOOLS = [symptom_severity_tool]
MEDICAL_RESEARCHER_TOOLS = [wikipedia_medical_lookup]

## 6) Agent Prompts (Role Separation)

This section specifies the system prompts that define each agent’s role and output contract. Clear prompts are critical in multi-agent design because they reduce overlap between responsibilities (extraction vs. research vs. summarization). The prompts also enforce structured outputs (JSON schemas and strict report format), which makes downstream parsing and routing reliable. In revision contexts, prompts serve as the conceptual “specification” of each agent’s behavior.

In [ ]:
SYMPTOM_EXTRACTOR_PROMPT = """You are a Symptom Analyzer Agent in an educational medical symptom assistant.

Your job:
1) Extract key symptoms, timing, and context from the user's message.
2) Identify any red flags (danger signs).
3) You MUST call the tool `symptom_severity_tool` at least once to estimate severity.

After you have the tool result, output ONLY valid JSON with this schema:
{
  "symptoms": [string],
  "duration": string,
  "age": number|null,
  "red_flags": [string],
  "patient_summary": string,
  "severity": "low"|"medium"|"high"|"emergency"
}

Notes:
- This is NOT real medical advice.
- Keep it concise.
"""

MEDICAL_RESEARCHER_PROMPT = """You are a Medical Research Agent in an educational medical symptom assistant.

Your job:
1) Use the tool `wikipedia_medical_lookup` to retrieve short background info relevant to the symptoms.
2) Propose a small list (2-4) of possible causes (not a diagnosis).
3) Summarize research in 2-4 short bullet points.

You MUST call `wikipedia_medical_lookup` at least once.
The query for `wikipedia_medical_lookup` MUST be maximum 4 words.

After you have tool results, output ONLY valid JSON with this schema:
{
  "possible_conditions": [string],
  "researcher_notes": string
}

Rules:
- Keep `possible_conditions` short and simple.
- Keep `researcher_notes` short (2-4 bullets max).
"""

REPORT_GENERATOR_PROMPT = """You are a Summary Agent in an educational medical symptom assistant.

Write a SHORT final recommendation for console output.

STRICT FORMAT (exactly this structure, no extra sections):
Patient Summary:
- Symptoms: <comma-separated>
- Possible Cause: <one item, not a diagnosis>
- Severity: low|medium|high|emergency
- Recommendation: <short>
- Suggested Action: <short>
- Disclaimer: Educational only, not medical advice.

Rules:
- 5-10 lines total.
- No long explanations.
- Do not repeat yourself.
"""

INTENT_GUARD_PROMPT = """You are an intent router for a medical symptom assistant.

Classify the user's message into ONE label:
- health: about symptoms, illness, injury, medical/health concerns, or asking what to do about health.
- non_health: greetings, chit-chat, coding, general knowledge, or anything not health-related.

Output ONLY one word: health OR non_health.
If uncertain, output non_health.
"""


## 7) Graph Nodes (Agent Calls, Parsing, and HITL)

This section implements the LangGraph nodes that execute the agent roles, handle tool routing, and post-process structured outputs. A dedicated interruption node introduces Human-in-the-Loop review as a controlled checkpoint before finalization. These node functions are the operational core of the graph and define how state evolves across steps.

In [ ]:
extractor_model = extractor_model.bind_tools(EXTRACTOR_TOOLS, tool_choice="auto")
medical_researcher_model= medical_researcher_model.bind_tools(MEDICAL_RESEARCHER_TOOLS, tool_choice="auto")

def intent_guard_node(state: MedicalState) -> Dict[str, Any]:
    user_request = state.get("user_request", "")
    resp = reporter_model.invoke([SystemMessage(INTENT_GUARD_PROMPT), HumanMessage(user_request)])
    label = (getattr(resp, "content", "") or "").strip().lower()
    intent: Literal["health", "non_health"] = "health" if label.startswith("health") else "non_health"
    return {
        "intent": intent,
        "log": [f"intent_guard: {intent}"],
    }


def non_medical_finalize_node(state: MedicalState) -> Dict[str, Any]:
    _ = state.get("user_request", "")  # mark state as used (lint)
    message = (
        "Sorry — I’m a medical symptom assistant and I can only help with health-related questions. "
        "Please describe any symptoms, how long they’ve been happening, and any red flags."
    )
    return {
        "messages": [AIMessage(content=message)],
        "final_recommendation": message,
        "log": ["intent_guard: non-medical request -> ended"],
    }


def route_after_intent_guard(state: MedicalState) -> str:
    return "symptom_extractor" if state.get("intent") == "health" else "non_medical_finalize"

def symptom_extractor_node(state: MedicalState) -> Dict[str, Any]:
    messages = state.get("messages", [])
    model = extractor_model

    response = model.invoke([SystemMessage(SYMPTOM_EXTRACTOR_PROMPT), *messages])
    return {
        "messages": [response],
        "log": ["symptom_analyzer: model call"],
    }


def medical_researcher_node(state: MedicalState) -> Dict[str, Any]:
    messages = state.get("messages", [])
    model = medical_researcher_model

    response = model.invoke([SystemMessage(MEDICAL_RESEARCHER_PROMPT), *messages])
    return {
        "messages": [response],
        "log": ["medical_research: model call"],
    }


def _format_recommendation_from_state(state: MedicalState) -> str:
    extracted_data = state.get("extracted_data", {})
    symptoms = extracted_data.get("symptoms", []) or []
    symptoms_text = ", ".join(symptoms) if symptoms else "not provided"

    possible_conditions = state.get("possible_conditions", []) or []
    possible_cause = possible_conditions[0] if possible_conditions else "unclear"

    severity = state.get("severity", "medium")

    if severity == "emergency":
        recommendation = "Seek emergency help now (call local emergency number)."
        suggested_action = "Do not drive yourself if unsafe; get urgent assistance."
    elif severity == "high":
        recommendation = "Get same-day medical evaluation (urgent care / doctor)."
        suggested_action = "If symptoms worsen or new red flags appear, seek urgent care."
    elif severity == "medium":
        recommendation = "Rest, hydrate, and monitor symptoms closely."
        suggested_action = "Contact a clinician if symptoms worsen or do not improve in 24-48h."
    else:
        recommendation = "Basic self-care and monitoring is reasonable."
        suggested_action = "If you feel worse or develop red flags, contact a clinician."

    return "\n".join(
        [
            "Patient Summary:",
            f"- Symptoms: {symptoms_text}",
            f"- Possible Cause: {possible_cause}",
            f"- Severity: {severity}",
            f"- Recommendation: {recommendation}",
            f"- Suggested Action: {suggested_action}",
            "- Disclaimer: Educational only, not medical advice.",
        ]
    )


def report_generator_node(state: MedicalState) -> Dict[str, Any]:

    user_request = state.get("user_request", "")

    extracted_data = state.get("extracted_data", {})

    possible_conditions = (
        state.get("possible_conditions", []) or []
    )[:3]

    researcher_notes = (
        state.get("researcher_notes", "") or ""
    ).strip()

    if len(researcher_notes) > 600:
        researcher_notes = researcher_notes[:600] + "…"

    compact_context = {
        "patient_summary": state.get("patient_summary", ""),
        "symptoms": extracted_data.get("symptoms", []),
        "duration": extracted_data.get("duration"),
        "age": extracted_data.get("age"),
        "red_flags": extracted_data.get("red_flags", []),
        "severity": state.get("severity"),
        "possible_conditions": possible_conditions,
        "researcher_notes": researcher_notes,
    }

    prompt_messages: List[BaseMessage] = [
        SystemMessage(REPORT_GENERATOR_PROMPT)
    ]

    if user_request:
        prompt_messages.append(
            HumanMessage(f"User request: {user_request}")
        )

    if (
        state.get("medical_expert_decision") == "revise"
        and state.get("medical_expert_guidance")
    ):
        prompt_messages.append(
            HumanMessage(
                "Medical expert revision guidance: "
                + state["medical_expert_guidance"]
            )
        )

    prompt_messages.append(
        HumanMessage(
            "Context JSON: "
            + json.dumps(compact_context, separators=(",", ":"))
        )
    )

    response = reporter_model.invoke(prompt_messages)

    draft = _cap_lines(
        getattr(response, "content", "") or "",
        max_lines=10,
    )

    if not draft.startswith("Patient Summary:"):
        draft = _format_recommendation_from_state(state)

    short_message = AIMessage(content=draft)

    return {
        "messages": [short_message],
        "medical_report": draft,
        "log": ["summary: draft created"],
    }


def has_pending_tool_calls(state: MedicalState) -> bool:
    messages = state.get("messages", [])
    if not messages:
        return False
    last_message = messages[-1]
    return isinstance(last_message, AIMessage) and last_message.tool_calls


def symptom_postprocess_node(state: MedicalState) -> Dict[str, Any]:
    messages = state.get("messages", [])
    last_ai_message = next((m for m in reversed(messages) if isinstance(m, AIMessage) and not m.tool_calls), None)
    if last_ai_message is None:
        return {"log": ["symptom_postprocess: no final AI message"]}

    data = _extract_json(last_ai_message.content)

    extracted_data = {
        "symptoms": data.get("symptoms", []),
        "duration": data.get("duration", "unknown"),
        "age": data.get("age"),
        "red_flags": data.get("red_flags", []),
    }

    return {
        "extracted_data": extracted_data,
        "patient_summary": data.get("patient_summary", ""),
        "severity": data.get("severity", "medium"),
        "log": ["symptom_postprocess: extracted structured data"],
    }


def medical_researcher_postprocess_node(state: MedicalState) -> Dict[str, Any]:
    messages = state.get("messages", [])
    last_ai_message = next((m for m in reversed(messages) if isinstance(m, AIMessage) and not m.tool_calls), None)
    if last_ai_message is None:
        return {"log": ["medical_researcher_postprocess: no final AI message"]}

    data = _extract_json(last_ai_message.content)

    possible_conditions = data.get("possible_conditions", [])
    if not isinstance(possible_conditions, list):
        possible_conditions = []

    return {
        "possible_conditions": possible_conditions,
        "researcher_notes": data.get("researcher_notes", ""),
        "log": ["medical_researcher_postprocess: parsed research JSON"],
    }

MAX_REVISIONS = 3

def medical_expert_review_interrupt_node(state: MedicalState) -> Dict[str, Any]:

    draft = state.get("medical_report", "")

    revision_count = state.get("revision_count", 0)

    if revision_count >= MAX_REVISIONS:
        return {
            "medical_expert_decision": "approve",
            "medical_expert_guidance": "",
            "log": [
                f"medical_expert_review: auto-approved after {MAX_REVISIONS} revisions"
            ],
        }

    prompt = (
        "Please review the following draft recommendation.\n\n"
        + draft
        + "\n\n"
        + "Return:\n"
        + "{decision: 'approve'|'revise', guidance: string}"
    )

    medical_expert_response = interrupt(prompt)
    if not medical_expert_response:
        return {
            "log": ["medical_expert_review: interrupted"]
        }

    decision = medical_expert_response.get("decision", "revise")
    guidance = medical_expert_response.get("guidance", "")

    if decision not in ["approve", "revise"]:
        decision = "revise"

    return {
        "medical_expert_decision": decision,
        "medical_expert_guidance": guidance,
        "revision_count": revision_count + 1,
        "log": [
            f"medical_expert_review: resumed with decision={decision}"
        ],
    }


def route_after_medical_expert_review(state: MedicalState) -> str:

    decision = state.get("medical_expert_decision", "revise")

    if decision == "approve":
        return "finalize"

    return "report_generator"


def finalize_node(state: MedicalState) -> Dict[str, Any]:
    final_text = state.get("medical_report", "")
    final_text = _cap_lines(final_text, max_lines=10)
    if not final_text.startswith("Patient Summary:"):
        final_text = _format_recommendation_from_state(state)

    return {
        "final_recommendation": final_text,
        "log": ["finalize: recommendation approved"],
    }



## 8) Graph Assembly and Visualization

This section wires the nodes into a directed stateful graph, including conditional routing for tool execution and HITL feedback loops. Explicit edges make the workflow’s control flow inspectable, which is useful for both debugging and exam discussion. A memory checkpointer is attached to preserve state across interruptions and resumes. Finally, the graph is visualized to provide a compact overview of the system architecture.

In [ ]:
checkpointer = InMemorySaver()

medical_builder = StateGraph(MedicalState)

medical_builder.add_node("intent_guard", intent_guard_node)
medical_builder.add_node("non_medical_finalize", non_medical_finalize_node)

medical_builder.add_node("symptom_extractor", symptom_extractor_node)
medical_builder.add_node("symptom_tools", ToolNode(EXTRACTOR_TOOLS))
medical_builder.add_node("symptom_postprocess", symptom_postprocess_node)

medical_builder.add_node("medical_researcher", medical_researcher_node)
medical_builder.add_node("medical_researcher_tools", ToolNode(MEDICAL_RESEARCHER_TOOLS))
medical_builder.add_node("medical_researcher_postprocess", medical_researcher_postprocess_node)

medical_builder.add_node("report_generator", report_generator_node)
medical_builder.add_node("medical_expert_review", medical_expert_review_interrupt_node)
medical_builder.add_node("finalize", finalize_node)

medical_builder.add_edge(START, "intent_guard")
medical_builder.add_conditional_edges(
    "intent_guard",
    route_after_intent_guard,
    ["symptom_extractor", "non_medical_finalize"],
)

medical_builder.add_conditional_edges(
    "symptom_extractor",
    lambda s: "symptom_tools" if has_pending_tool_calls(s) else "symptom_postprocess",
    ["symptom_tools", "symptom_postprocess"],
)
medical_builder.add_edge("symptom_tools", "symptom_extractor")
medical_builder.add_edge("symptom_postprocess", "medical_researcher")

medical_builder.add_conditional_edges(
    "medical_researcher",
    lambda s: "medical_researcher_tools" if has_pending_tool_calls(s) else "medical_researcher_postprocess",
    ["medical_researcher_tools", "medical_researcher_postprocess"],
)
medical_builder.add_edge("medical_researcher_tools", "medical_researcher")
medical_builder.add_edge("medical_researcher_postprocess", "report_generator")
medical_builder.add_edge("report_generator", "medical_expert_review")

medical_builder.add_conditional_edges(
    "medical_expert_review",
    route_after_medical_expert_review,
    ["finalize", "report_generator"],
)

medical_builder.add_edge("non_medical_finalize", END)
medical_builder.add_edge("finalize", END)

medical_graph = medical_builder.compile(checkpointer=checkpointer)

display_graph(medical_graph, Path("/content/medical_graph.png"))

## 9) Workflow Entry Point (Execution and Resume)

This section provides the public function used to run the compiled graph from a single user request. The wrapper is responsible for initializing the thread/configuration, handling HITL interruptions, and resuming execution with human feedback. Encapsulating execution logic behind a single function makes the notebook easier to test and demonstrates a clean API boundary for the exam requirements. The optional printing controls support both concise runs and full trace-style review of agent messages.

In [ ]:
# Optional: automated decisions for tests (each item is {decision, guidance}).
# If empty/None, the notebook will ask via input().
MEDICAL_EXPERT_DECISIONS_PRESET: Optional[List[Dict[str, str]]] = None


def _get_next_medical_expert_decision() -> Dict[str, str]:
    """
    Single-input HITL parser.

    Rules:
    - If the user types 'approve' (or 'ok') in ANY casing, treat as approval.
    - Any other non-empty text is treated as revision guidance.

    Examples:
        approve
        OK
        Make emergency warning stronger
    """

    global MEDICAL_EXPERT_DECISIONS_PRESET

    if (
        MEDICAL_EXPERT_DECISIONS_PRESET is not None
        and len(MEDICAL_EXPERT_DECISIONS_PRESET) > 0
    ):
        return MEDICAL_EXPERT_DECISIONS_PRESET.pop(0)

    raw = input(
        "\nMedical Expert Review\n"
        "----------------------\n"
        "Type ONE of the following:\n\n"
        "approve (or ok)\n"
        "<anything else> = revise with that text as guidance\n\n"
        "Your input: "
    ).strip()

    normalized = raw.strip().lower()
    approve_words = {"approve", "approved", "ok", "okay"}

    if normalized in approve_words:
        return {
            "decision": "approve",
            "guidance": "",
        }

    guidance = raw.strip()
    if not guidance:
        guidance = "Please improve clarity and safety guidance."

    return {
        "decision": "revise",
        "guidance": guidance,
    }


def execute_workflow(
    user_request: str,
    print_whole_conversation: bool = False,
) -> str:

    thread_id = f"medical_{uuid.uuid4().hex[:8]}"

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    state: MedicalState = medical_graph.invoke(
        input={
            "user_request": user_request,
            "messages": [HumanMessage(user_request)],
            "log": [],
            "revision_count": 0,
        },
        config=config,
    )
    while state.get("__interrupt__"):

        interrupts = state["__interrupt__"]

        print_interrupts(interrupts)

        decision_payload = _get_next_medical_expert_decision()

        resume_payload = {
            interrupt_obj.id: decision_payload
            for interrupt_obj in interrupts
        }

        state = medical_graph.invoke(
            input=Command(resume=resume_payload),
            config=config,
        )

    if print_whole_conversation:

        print("\n\n===== FULL CONVERSATION =====")

        print_conversation(
            state.get("messages", [])
        )


    final_output = state.get(
        "final_recommendation",
        "No recommendation generated.",
    )

    print("\n--- FINAL RECOMMENDATION (educational only) ---\n")

    return print(final_output)


## 10) Tests

With the default models it should take around 3-4 minutes.

In [ ]:
tests = [
    {
        "name": "Non-Health intent",
        "request": "Hello, how are you?",
        "decisions": [
            {
                "decision": "",
                "guidance": ""
            }
        ],
    },
    {
        "name": "Headache + fever (approve)",
        "request": "I have had a headache and fever for 2 days. I feel tired and my throat is sore. What should I do?",
        "decisions": [
            {
                "decision": "approve",
                "guidance": ""
            }
        ],
    },
    {
        "name": "Chest pain (revise -> approve)",
        "request": "I have chest pain and shortness of breath that started suddenly 30 minutes ago. I'm sweating.",
        "decisions": [
            {
                "decision": "revise",
                "guidance": "Make the emergency advice more prominent and add a short list of immediate actions."
            },
            {
                "decision": "approve",
                "guidance": ""
            },
        ],
    },
    {
        "name": "Stomach ache (approve)",
        "request": "I have stomach pain and vomiting since last night. I can keep small sips of water. No blood.",
        "decisions": [
            {
                "decision": "approve",
                "guidance": ""
            }
        ],
    },
    {
        "name": "Allergic reaction (approve)",
        "request": "I developed hives after eating shrimp. My lips feel a bit swollen but I can breathe ok.",
        "decisions": [
            {
                "decision": "approve",
                "guidance": ""
            }
        ],
    },
    {
        "name": "Dizziness (approve)",
        "request": "I've been dizzy all day and nearly fainted when standing up. I haven't eaten much today.",
        "decisions": [
            {
                "decision": "approve",
                "guidance": ""
            }
        ],
    },
    {
        "name": "Severe dehydration (revise -> revise -> approve)",
        "request": "I've had diarrhea for 3 days and now I feel extremely weak and dizzy. My mouth is very dry and I barely urinated today.",
        "decisions": [
            {
                "decision": "revise",
                "guidance": "Emphasize dehydration danger signs and recommend urgent medical evaluation more clearly."
            },
            {
                "decision": "revise",
                "guidance": "Add immediate self-care advice such as oral rehydration and warning symptoms that require emergency care."
            },
            {
                "decision": "approve",
                "guidance": ""
            },
        ],
    },
    {
        "name": "Persistent cough (approve)",
        "request": "I've had a cough, mild fever, and congestion for about 5 days. I feel tired but I'm breathing normally.",
        "decisions": [
            {
                "decision": "approve",
                "guidance": ""
            }
        ],
    }
]

for t in tests:
    print("\n\n" + "#" * 70)
    print(t["name"] + "(Test scenario)")
    print("#" * 70)

    MEDICAL_EXPERT_DECISIONS_PRESET = list(t["decisions"])
    conversation = execute_workflow(t["request"], print_whole_conversation=True)


In [ ]:
execute_workflow("If you are not a hypochondriac, try it yourself! Thanks!")